In [1]:
import xarray as xr

In [8]:
data = xr.open_zarr(
    "/scratch/as15415/Data/Emulation_Data/OM4_Horizontal_Regrid_Old.zarr"
)

# data = xr.open_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/test_CMIP6_GFDL-CM4.piControl.r1i1p1f1.zarr")

In [9]:
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

### Convert and save surface data to /vast

Make sure you are using atleast 8 cores!

In [10]:
import sys

sys.path.append("../src/")

In [11]:
from utils.data_utils import get_wet_mask

In [14]:
inputs = []
inputs.append(data["uo"][:, 0])
inputs.append(data["vo"][:, 0])
inputs.append(data["thetao"][:, 0])
inputs.append(data["so"][:, 0])
inputs.append(data["zos"])
inputs = tuple(inputs)
wet, _ = get_wet_mask(inputs)
print("Wet resolution:", wet.shape)

Wet resolution: torch.Size([180, 360])


In [14]:
# print("Calculating mask tensors")
# wet, wet_nan = get_wet_mask(inputs, "cpu")
# # wet_bool = np.array(wet.cpu()).astype(bool)
# # wet_lap = compute_laplacian_wet(wet_nan, 4)  # hardcoded
# # wet_lap = xr.where(wet_lap == 0, 1, np.nan)
# # wet_lap = np.nan_to_num(wet_lap)
# print("Wet resolution:", wet.shape)

Calculating mask tensors
Wet resolution: torch.Size([180, 360])


In [16]:
import torch

In [20]:
torch.save(wet, "/vast/sd5313/data/m2lines/3D_ocean_data/surface_wet.pt")

In [5]:
from dask.diagnostics import ProgressBar

In [6]:
with ProgressBar():
    data_mean = data.mean().compute()

[########################################] | 100% Completed | 259.26 s


In [7]:
data_mean.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_means")

In [8]:
with ProgressBar():
    data_std = data.std().compute()

[########################################] | 100% Completed | 244.16 s


In [9]:
data_std.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_stds")

In [10]:
data.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data")

### Convert and save 3D data to /vast

Make sure you are using atleast 10 cores!

In [5]:
from dask.diagnostics import ProgressBar

In [6]:
ds = data

In [7]:
for lev in ds["lev"].values:
    lev_str = str(lev).replace(".", "_")

    # Create new variables for each original variable with the lev dimension
    ds[f"vo_lev_{lev_str}"] = ds["vo"].sel(lev=lev)
    ds[f"thetao_lev_{lev_str}"] = ds["thetao"].sel(lev=lev)
    ds[f"uo_lev_{lev_str}"] = ds["uo"].sel(lev=lev)
    ds[f"so_lev_{lev_str}"] = ds["so"].sel(lev=lev)

ds = ds.drop_vars(["vo", "thetao", "uo", "so"])
ds

<xarray.Dataset>
Dimensions:            (time: 6000, y: 154, x: 360, lev: 35)
Coordinates:
    lat                (y, x) float64 dask.array<chunksize=(77, 360), meta=np.ndarray>
  * lev                (lev) float64 2.5 10.0 20.0 ... 5.5e+03 6e+03 6.5e+03
    lon                (y, x) float64 dask.array<chunksize=(77, 360), meta=np.ndarray>
    sub_experiment_id  <U4 ...
  * time               (time) object 0151-01-16 12:00:00 ... 0650-12-16 12:00:00
    variant_label      <U8 ...
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.5 -88.5 -87.5 -86.5 ... 61.5 62.5 63.5
Data variables: (12/142)
    hft                (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    zos                (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    vo_lev_2_5         (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    thetao_lev_2_5     (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    uo_lev_2_5         (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    so_lev_2_5         (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    ...                 ...
    uo_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    so_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    vo_lev_6500_0      (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    thetao_lev_6500_0  (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    uo_lev_6500_0      (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
    so_lev_6500_0      (time, y, x) float32 dask.array<chunksize=(1, 154, 360), meta=np.ndarray>
Attributes: (12/49)
    Conventions:                         CF-1.7 CMIP-6.0 UGRID-1.0
    activity_id:                         CMIP
    branch_method:                       standard
    branch_time_in_child:                0.0
    branch_time_in_parent:               54750.0
    comment:                             <null ref>
    ...                                  ...
    sub_experiment:                      none
    sub_experiment_id:                   none
    title:                               NOAA GFDL GFDL-CM4 model output prep...
    variant_info:                        N/A
    variant_label:                       r1i1p1f1
    version_id:                          v20180701

In [8]:
with ProgressBar():
    ds_mean = ds.mean().compute()

[########################################] | 100% Completed | 12m 35s


In [9]:
ds_mean.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_CM4_means")

In [10]:
with ProgressBar():
    ds_std = ds.std().compute()

[########################################] | 100% Completed | 16m 20s


In [11]:
ds_std.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_CM4_stds")

In [12]:
with ProgressBar():
    ds.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_CM4")

[########################################] | 100% Completed | 18m 11s


#### Wet mask

In [1]:
import xarray as xr

In [5]:
data = xr.open_zarr("/pscratch/sd/s/suryad/data/OM4_Horizontal_Regrid_Old.zarr")
levels = 5
# data = xr.open_zarr(
#     "/vast/sd5313/data/m2lines/3D_ocean_data/test_CMIP6_GFDL-CM4.piControl.r1i1p1f1.zarr"
# )
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [6]:
data = data.drop(["tauuo", "tauvo", "hfds"])
# data = data.drop(["tauuo", "tauvo", "hft"])
data

<xarray.Dataset>
Dimensions:  (time: 4745, lev: 19, y: 180, x: 360)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [7]:
import numpy as np
import torch


def get_wet_mask(inputs, device="cpu"):
    wet = xr.zeros_like(inputs[0][0])
    # inputs[0][0,12,12] = np.nan
    for data in inputs:
        wet += np.isnan(data[0])

    wet_nan = xr.where(wet != 0, np.nan, 1).to_numpy()
    wet = np.isnan(xr.where(wet == 0, np.nan, 0))
    wet = np.nan_to_num(wet.to_numpy())
    wet = torch.from_numpy(wet).type(torch.float32).to(device=device)
    return wet, wet_nan

In [8]:
wet_stacked = []
for i in range(levels):
    inputs = []
    inputs.append(data["uo"][:, i])
    inputs.append(data["vo"][:, i])
    inputs.append(data["thetao"][:, i])
    inputs.append(data["so"][:, i])
    if i == 0:
        inputs.append(data["zos"])

    inputs = tuple(inputs)
    wet, _ = get_wet_mask(inputs)
    wet_stacked.append(wet)

In [9]:
wet_3D = torch.stack(wet_stacked)

In [10]:
out_list = [
    k + str(j)
    for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
    for j in range(levels)
] + ["zos"]
out_list

['uo_lev_0',
 'uo_lev_1',
 'uo_lev_2',
 'uo_lev_3',
 'uo_lev_4',
 'vo_lev_0',
 'vo_lev_1',
 'vo_lev_2',
 'vo_lev_3',
 'vo_lev_4',
 'thetao_lev_0',
 'thetao_lev_1',
 'thetao_lev_2',
 'thetao_lev_3',
 'thetao_lev_4',
 'so_lev_0',
 'so_lev_1',
 'so_lev_2',
 'so_lev_3',
 'so_lev_4',
 'zos']

In [11]:
final_wet = []
for var in out_list:
    try:
        level = int(var.split("lev_")[-1])
    except:
        level = 0
    final_wet.append(wet_3D[level])

In [12]:
wet = torch.stack(final_wet)

In [16]:
assert wet.shape[0] == (levels * 4 + 1)

In [17]:
torch.save(wet, "/pscratch/sd/s/suryad/data/3D_wet_OM4_{0}_levels.pt".format(levels))

### Tests

In [3]:
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [4]:
data.so["lev"]

<xarray.DataArray 'lev' (lev: 19)>
array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18])
Dimensions without coordinates: lev

In [14]:
def test_gen(input_vars, extra_vars, output_vars, lag=1, depth_mode="all"):
    data = xr.open_zarr(
        "/scratch/as15415/Data/Emulation_Data/OM4_Horizontal_Regrid_Old.zarr"
    )

    inputs = []
    extra_in = []
    outputs = []

    for var in input_vars:
        if var == "zos":
            inputs.append(data[var])
        elif depth_mode == "surface":
            inputs.append(data[var][:, 0])
        elif depth_mode == "all":
            for i in range(data[var].shape[1]):
                inputs.append(data[var][:, i])

    for var in extra_vars:
        extra_in.append(data[var])

    for var in output_vars:
        if var == "zos":
            outputs.append(data[var][lag:])
        elif depth_mode == "surface":
            outputs.append(data[var][lag:, 0])
        elif depth_mode == "all":
            for i in range(data[var].shape[1]):
                outputs.append(data[var][lag:, i])

    inputs = tuple(inputs)
    extra_in = tuple(extra_in)
    outputs = tuple(outputs)

    return inputs, extra_in, outputs

In [17]:
inputs, extra_in, outputs = test_gen(
    ["uo", "vo", "thetao", "so", "zos"],
    ["tauuo", "tauvo", "hfds"],
    ["uo", "vo", "thetao", "so", "zos"],
    depth_mode="all",
)

In [20]:
len(inputs)

77

In [8]:
import numpy as np

In [9]:
data.tauuo[:, :, :].values

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [ 0.03739167,  0.03725152,  0.03707196, ...,  0.03757205,
          0.03754889,  0.03749075],
        [ 0.02840508,  0.02819717,  0.02796913, ...,  0.02895568,
          0.02878909,  0.02860614],
        [ 0.01770086,  0.01757012,  0.01743707, ...,  0.01812409,
          0.01798942,  0.01784848]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.11497339, -0.11504205, -0.1151281 , ..., -

In [10]:
np.array(data.zos)

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.32888971, -0.32703181, -0.32515863, ..., -0.33432069,
         -0.33258432, -0.33075901],
        [-0.29781562, -0.29685536, -0.29586226, ..., -0.30082796,
         -0.29984848, -0.29884841],
        [-0.2679397 , -0.26772973, -0.26750271, ..., -0.2687237 ,
         -0.2684744 , -0.26819819]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.59817378, -0.59610624, -0.59397595, ..., -

In [11]:
data.uo[0].values

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.01251889, -0.01230785, -0.01209403, ..., -0.01272803,
         -0.01274474, -0.01267345],
        [-0.01845972, -0.01802384, -0.0176047 , ..., -0.01959374,
         -0.01925796, -0.01886955],
        [-0.02259186, -0.02249625, -0.02240956, ..., -0.02279709,
         -0.02275002, -0.02269251]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.0302047 , -0.03006262, -0.02988526, ..., -